# Chẩn đoán quá trình train RQ-VAE

Notebook này so sánh hai cách khởi tạo trên cùng một sample embedding:

- **Baseline:** KMeans chạy ngay trên latent của encoder ngẫu nhiên, giống luồng train hiện tại.
- **AE trước:** pretrain autoencoder, sau đó mới khởi tạo KMeans và train RQ-VAE.

Mục tiêu là kiểm tra quantization loss tăng do dữ liệu hay do codebook được khởi tạo trước khi encoder ổn định. Notebook không sửa `src` và không tạo checkpoint dùng cho production.

## 1. Cấu hình

Đặt `EMBEDDING_ROOT` nếu muốn chỉ định trực tiếp Kaggle Dataset. Để `None` thì notebook tự tìm output của notebook 02 trong `/kaggle/input`.

In [ ]:
from pathlib import Path
import importlib.util
import os
import random
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

EMBEDDING_ROOT = None
GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")

SEED = 2026
SAMPLE_SIZE = 50_000
INIT_SIZE = 20_000
EVAL_SIZE = 8_192
BATCH_SIZE = 512
AE_STEPS = 500
RQVAE_STEPS = 1_000
LOG_EVERY = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

## 2. Chuẩn bị source trên Kaggle

Notebook cài các dependency còn thiếu và clone source từ nhánh `main`. Cần bật Kaggle Secret `GITHUB_TOKEN`.

In [ ]:
required_modules = {
    "gin": "gin-config==0.5.0",
    "einops": "einops>=0.8.0",
    "huggingface_hub": "huggingface-hub>=0.25.0",
}
missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
git_environment = {**os.environ, "GITHUB_TOKEN": github_token, "GIT_TERMINAL_PROMPT": "0"}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
        env=git_environment,
    )
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
        env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation" / "src"
if not (SOURCE_ROOT / "modules" / "rqvae.py").is_file():
    raise FileNotFoundError(f"RQ-VAE source not found: {SOURCE_ROOT}")
sys.path.insert(0, str(SOURCE_ROOT))
print("SOURCE_ROOT:", SOURCE_ROOT)

## 3. Tìm output của notebook 02

Notebook tự tìm hai file `global_product_embeddings.f16.npy` và `global_embedding_index.parquet` trong Kaggle Dataset đã được add vào notebook.

In [ ]:
def is_embedding_root(path):
    path = Path(path)
    return (
        (path / "global_product_embeddings.f16.npy").is_file()
        and (path / "global_embedding_index.parquet").is_file()
    )


def locate_embedding_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_embedding_root(root):
            return root
        raise FileNotFoundError(f"Notebook 02 artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/embeddings"),
        cwd / "embeddings",
        cwd.parent / "embeddings",
    ]
    for candidate in candidates:
        if is_embedding_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for manifest_path in kaggle_input.glob("**/embedding_manifest.json"):
            if is_embedding_root(manifest_path.parent):
                return manifest_path.parent.resolve()
        for embedding_path in kaggle_input.glob("**/global_product_embeddings.f16.npy"):
            if is_embedding_root(embedding_path.parent):
                return embedding_path.parent.resolve()

    raise FileNotFoundError(
        "Notebook 02 output was not found. Add it as a Kaggle Dataset or set EMBEDDING_ROOT."
    )


EMBEDDING_ROOT = locate_embedding_root(EMBEDDING_ROOT)
EMBEDDING_PATH = EMBEDDING_ROOT / "global_product_embeddings.f16.npy"
print("EMBEDDING_ROOT:", EMBEDDING_ROOT)

## 4. Đọc sample cố định

Hai thí nghiệm dùng cùng dữ liệu và cùng seed để kết quả có thể so sánh trực tiếp.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

embedding_matrix = np.load(EMBEDDING_PATH, mmap_mode="r")
rng = np.random.default_rng(SEED)
sample_size = min(SAMPLE_SIZE, len(embedding_matrix))
sample_rows = rng.choice(len(embedding_matrix), size=sample_size, replace=False)
sample = torch.from_numpy(
    np.asarray(embedding_matrix[sample_rows], dtype=np.float32).copy()
)

eval_size = min(EVAL_SIZE, len(sample) // 5)
eval_data = sample[:eval_size]
train_data = sample[eval_size:]
init_data = train_data[: min(INIT_SIZE, len(train_data))].to(DEVICE)

print("Full matrix:", embedding_matrix.shape, embedding_matrix.dtype)
print("Train sample:", tuple(train_data.shape))
print("Evaluation sample:", tuple(eval_data.shape))
print("KMeans initialization sample:", tuple(init_data.shape))

## 5. Hàm train và đo lường

Các metric được tính bằng hard nearest-code trên evaluation sample:

- `reconstruction_loss`: khoảng cách giữa embedding gốc và embedding tái tạo.
- `quantization_loss`: tổng loss của ba tầng codebook.
- `layer_*_loss`: quantization loss riêng từng tầng.
- `layer_*_usage`: tỷ lệ code được dùng trong 256 code.
- `layer_*_residual_norm`: norm đầu vào của từng tầng.

In [ ]:
from modules.quantize import QuantizeForwardMode
from modules.rqvae import RqVae


def build_model():
    torch.manual_seed(SEED)
    return RqVae(
        input_dim=256,
        embed_dim=32,
        hidden_dims=[256, 128, 64],
        codebook_size=256,
        codebook_kmeans_init=True,
        codebook_normalize=False,
        codebook_sim_vq=False,
        codebook_mode=QuantizeForwardMode.STE,
        n_layers=3,
        commitment_weight=0.25,
        n_cat_features=0,
    ).to(DEVICE)


def next_batch(data, generator):
    rows = torch.randint(len(data), (BATCH_SIZE,), generator=generator)
    return data[rows].to(DEVICE)


def rqvae_objective(model, x):
    residual = model.encode(x)
    embeddings = []
    layer_losses = []
    layer_ids = []
    residual_norms = []

    for layer in model.layers:
        residual_norms.append(residual.norm(dim=-1).mean())
        quantized = layer(residual, temperature=0.2)
        embeddings.append(quantized.embeddings)
        layer_losses.append(quantized.loss.mean())
        layer_ids.append(quantized.ids)
        residual = residual - quantized.embeddings

    reconstructed = model.decode(torch.stack(embeddings).sum(dim=0))
    reconstruction_loss = ((reconstructed - x) ** 2).sum(dim=-1).mean()
    quantization_loss = torch.stack(layer_losses).sum()

    return {
        "total_loss": reconstruction_loss + quantization_loss,
        "reconstruction_loss": reconstruction_loss,
        "quantization_loss": quantization_loss,
        "layer_losses": layer_losses,
        "layer_ids": layer_ids,
        "residual_norms": residual_norms,
    }

In [ ]:
@torch.no_grad()
def initialize_codebooks(model, x):
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    model.eval()
    residual = model.encode(x)

    for layer_index, layer in enumerate(model.layers):
        layer._kmeans_init(residual)
        quantized = layer(residual, temperature=0.2)
        residual = residual - quantized.embeddings
        print(
            f"Layer {layer_index}: initialized 256 codes, "
            f"remaining residual norm={residual.norm(dim=-1).mean().item():.6f}"
        )


@torch.no_grad()
def evaluate(model, data, run_name, step):
    model.eval()
    parts = []

    for start in range(0, len(data), BATCH_SIZE):
        parts.append(rqvae_objective(model, data[start : start + BATCH_SIZE].to(DEVICE)))

    row = {
        "run": run_name,
        "step": step,
        "total_loss": np.mean([part["total_loss"].item() for part in parts]),
        "reconstruction_loss": np.mean(
            [part["reconstruction_loss"].item() for part in parts]
        ),
        "quantization_loss": np.mean(
            [part["quantization_loss"].item() for part in parts]
        ),
    }

    for layer_index in range(len(model.layers)):
        ids = torch.cat([part["layer_ids"][layer_index].cpu() for part in parts])
        row[f"layer_{layer_index}_loss"] = np.mean(
            [part["layer_losses"][layer_index].item() for part in parts]
        )
        row[f"layer_{layer_index}_usage"] = (
            torch.unique(ids).numel() / model.layers[layer_index].n_embed
        )
        row[f"layer_{layer_index}_residual_norm"] = np.mean(
            [part["residual_norms"][layer_index].item() for part in parts]
        )

    return row


def pretrain_autoencoder(model, data):
    optimizer = torch.optim.AdamW(
        list(model.encoder.parameters()) + list(model.decoder.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    generator = torch.Generator().manual_seed(SEED + 1)
    history = []

    for step in range(1, AE_STEPS + 1):
        model.train()
        x = next_batch(data, generator)
        reconstructed = model.decode(model.encode(x))
        loss = ((reconstructed - x) ** 2).sum(dim=-1).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step == 1 or step % LOG_EVERY == 0:
            history.append({"step": step, "reconstruction_loss": loss.item()})
            print(f"AE step {step:4d} | reconstruction={loss.item():.6f}")

    return pd.DataFrame(history)


def train_rqvae(model, data, eval_data, run_name):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    generator = torch.Generator().manual_seed(SEED + 2)
    history = [evaluate(model, eval_data, run_name, step=0)]

    for step in range(1, RQVAE_STEPS + 1):
        model.train()
        x = next_batch(data, generator)
        output = rqvae_objective(model, x)

        optimizer.zero_grad()
        output["total_loss"].backward()
        optimizer.step()

        if step % LOG_EVERY == 0:
            row = evaluate(model, eval_data, run_name, step)
            history.append(row)
            print(
                f"{run_name:10s} step {step:4d} | "
                f"reconstruction={row['reconstruction_loss']:.6f} | "
                f"quantization={row['quantization_loss']:.6f}"
            )

    return pd.DataFrame(history)

## 6. Baseline: KMeans trước khi encoder được train

Đây là thứ tự hiện tại trong `train_rqvae.py`.

In [ ]:
baseline_model = build_model()
initialize_codebooks(baseline_model, init_data)
baseline_history = train_rqvae(
    baseline_model, train_data, eval_data, run_name="baseline"
)

## 7. Pretrain autoencoder rồi mới KMeans

Codebook chỉ được khởi tạo sau khi không gian latent của encoder đã học được khả năng tái tạo embedding.

In [ ]:
staged_model = build_model()
ae_history = pretrain_autoencoder(staged_model, train_data)
initialize_codebooks(staged_model, init_data)
staged_history = train_rqvae(
    staged_model, train_data, eval_data, run_name="ae_then_kmeans"
)

## 8. So sánh kết quả

Quan sát quan trọng nhất là đường `quantization_loss`. Nếu nhánh `ae_then_kmeans` ổn định hơn rõ rệt, thứ tự khởi tạo là nguyên nhân chính.

In [ ]:
history = pd.concat([baseline_history, staged_history], ignore_index=True)
display(history.groupby("run").agg(
    initial_total_loss=("total_loss", "first"),
    final_total_loss=("total_loss", "last"),
    initial_reconstruction_loss=("reconstruction_loss", "first"),
    final_reconstruction_loss=("reconstruction_loss", "last"),
    initial_quantization_loss=("quantization_loss", "first"),
    final_quantization_loss=("quantization_loss", "last"),
))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for run_name, frame in history.groupby("run"):
    axes[0, 0].plot(frame["step"], frame["reconstruction_loss"], label=run_name)
    axes[0, 1].plot(frame["step"], frame["quantization_loss"], label=run_name)
    for layer_index in range(3):
        label = f"{run_name} / layer {layer_index}"
        axes[1, 0].plot(
            frame["step"], frame[f"layer_{layer_index}_loss"], label=label
        )
        axes[1, 1].plot(
            frame["step"], frame[f"layer_{layer_index}_usage"], label=label
        )

axes[0, 0].set_title("Reconstruction loss")
axes[0, 1].set_title("Quantization loss")
axes[1, 0].set_title("Quantization loss theo tầng")
axes[1, 1].set_title("Codebook usage theo tầng")

for axis in axes.flat:
    axis.set_xlabel("RQ-VAE step")
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
columns = [
    "run",
    "step",
    "total_loss",
    "reconstruction_loss",
    "quantization_loss",
    "layer_0_loss",
    "layer_1_loss",
    "layer_2_loss",
    "layer_0_usage",
    "layer_1_usage",
    "layer_2_usage",
    "layer_0_residual_norm",
    "layer_1_residual_norm",
    "layer_2_residual_norm",
]
display(history[columns].tail(6))

## 9. Cách đọc kết quả

- Nếu baseline tăng quantization loss nhưng `ae_then_kmeans` không tăng: chuyển pipeline chính sang pretrain AE trước KMeans.
- Nếu cả hai cùng tăng: kiểm tra tiếp STE và gradient qua residual.
- Nếu usage của một tầng giảm mạnh: tầng đó đang collapse.
- Nếu usage vẫn cao nhưng loss tăng: codebook còn đa dạng nhưng không theo kịp latent hoặc scale của residual.

Không dùng model của notebook này làm checkpoint chính; đây chỉ là controlled experiment.